# Lesson 4: Memory

**Project:** Agentic Engineering Worklog
**Model:** `qwen3:8b`, running on this PC through Ollama

This lesson gives the agent a memory of the conversation, so it can follow a back-and-forth. It also produced the most surprising results so far. **Several things expected to fail without memory worked anyway**, and working out why turned out to be the most useful part of the lesson. The explanations below describe what actually happened in this run, not what was predicted.

## Where lesson 3 left off

Lesson 3's agent scored 18 of 18, but every request started from nothing. As soon as it answered, it forgot the conversation. So a follow-up like this was expected to fail:

> **You:** Log an hour on Herd Planner fixing the weight chart.
>
> **You:** Actually, make that two hours.

"That" refers to the first message, which the second request never sees.

## The big idea: what is "memory" for an agent?

Lesson 1 showed that **a model has no memory of its own.** Each call starts fresh, and it knows only what's in the message list you send.

So "memory" for an agent is something **your code** does: it keeps the earlier messages and sends them again with each new request. The model reads the whole conversation every time.

That has a cost. Every remembered message is re-read on every call, which makes calls slower (and, with a paid service, more expensive). So memory needs a **limit**: keep the recent part of the conversation, and drop the oldest.

There's a second kind of memory, which this lesson's results bring out: **saved data.** The work log file remembers every entry, whether or not the conversation does. A tool that reads the file can recover what happened, even with no conversation memory at all.

## What we are looking for

By the end of this notebook you should be able to say yes to five things:

1. I understand that conversation memory is just keeping earlier messages and sending them again.
2. I have a third tool, `update_last_entry`, so a correction changes an entry instead of adding a duplicate.
3. I can tell the difference between memory in the conversation and memory in saved data.
4. I know why conversation memory needs a limit, and what it costs.
5. I can check whether a test really tests what it claims to.

**How to run it in PyCharm:** click a code cell and press **Shift + Enter**. Run the cells in order from top to bottom.

## Step 1: Setup

### What we are looking for

A separate **demo log** with four known entries, reset before every demonstration, so each step starts from the same place and your real work log is never touched.

In [1]:
import json
import re
from datetime import date, datetime, timedelta
from pathlib import Path

import ollama

MODEL = "qwen3:8b"

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

DATA_DIR = project_root / "data"
DATA_DIR.mkdir(exist_ok=True)

DEMO_LOG = DATA_DIR / "demo_log_lesson4.jsonl"
LOG_FILE = DEMO_LOG  # the file the tools read and write

TODAY = date.today()
TODAY_S = TODAY.isoformat()
YESTERDAY = (TODAY - timedelta(days=1)).isoformat()

STARTING_ENTRIES = [
    {"logged_at": f"{YESTERDAY}T09:00:00", "project": "Cornerpost Watch", "description": "Tuned the cloud mask", "hours": 1},
    {"logged_at": f"{YESTERDAY}T14:00:00", "project": "Agentic Engineering Worklog", "description": "Ran lesson 3", "hours": None},
    {"logged_at": f"{TODAY_S}T08:30:00", "project": "Cornerpost Watch", "description": "Fixed the alert email bug", "hours": 2},
    {"logged_at": f"{TODAY_S}T10:00:00", "project": "Herd Planner", "description": "Added vaccine records", "hours": 1.5},
]


def reset_log():
    """Put the demo log back to the four starting entries."""
    LOG_FILE.write_text("".join(json.dumps(e) + "\n" for e in STARTING_ENTRIES), encoding="utf-8")


reset_log()
print("Demo log:", LOG_FILE)
print("Today:   ", TODAY_S)
print("Entries: ", len(STARTING_ENTRIES))

Demo log: B:\_Dev\Python\Agentic-Engineering-Worklog\data\demo_log_lesson4.jsonl
Today:    2026-11-16
Entries:  4


### The code, line by line

- The imports and paths are lesson 3's.
- `DEMO_LOG` is a separate file just for this lesson, and `LOG_FILE = DEMO_LOG` points every tool at it.
- `TODAY_S` and `YESTERDAY` are today's and yesterday's dates as text, such as `2026-11-16`. `timedelta(days=1)` means "one day".
- `STARTING_ENTRIES` are the same four entries lesson 3's test used:

| When | Project | Description | Hours |
|---|---|---|---|
| yesterday | Cornerpost Watch | Tuned the cloud mask | 1 |
| yesterday | Agentic Engineering Worklog | Ran lesson 3 | none |
| today | Cornerpost Watch | Fixed the alert email bug | 2 |
| today | Herd Planner | Added vaccine records | 1.5 |

- `reset_log()` rewrites the demo log with exactly these four entries.

### How to read it

Your run printed the demo log's path (ending in `data\demo_log_lesson4.jsonl`), `Today: 2026-11-16`, and `Entries: 4`.

*Since this run, the projects moved from `B:\_Dev\Python` to `B:\Development\Python`. The saved output above still shows the old path, because it's a record of what ran. Running the cell again would show the new one.*

### What it means

Knowing exactly what's in the log makes every result in this lesson checkable by hand. For example, Herd Planner today starts at **1.5 hours**, so any answer about Herd Planner can be worked out from there.

### What's next and why

Corrections need a tool that changes an entry. Step 2 adds it.

## Step 2: Three tools as plain Python

### What we are looking for

The new tool, `update_last_entry`, working by hand. `add_entry` and `search_entries` are unchanged from lesson 3.

In [2]:
def read_entries():
    """Every entry in the current log file, oldest first."""
    if not LOG_FILE.exists():
        return []
    lines = LOG_FILE.read_text(encoding="utf-8").splitlines()
    return [json.loads(line) for line in lines if line.strip()]


def write_entries(entries):
    """Replace the log file with these entries."""
    LOG_FILE.write_text("".join(json.dumps(e) + "\n" for e in entries), encoding="utf-8")


def add_entry(project, description, hours=None):
    """Save one work log entry and return a short confirmation."""
    entry = {
        "logged_at": datetime.now().isoformat(timespec="seconds"),
        "project": project,
        "description": description,
        "hours": hours,
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")
    return f"Saved entry for {project}."


def search_entries(project=None, date_from=None, date_to=None):
    """Find entries by project and date range (YYYY-MM-DD, both ends included). Returns text with a total."""
    matches = []
    for entry in read_entries():
        day = entry["logged_at"][:10]
        if project and entry["project"].lower() != project.lower():
            continue
        if date_from and day < date_from:
            continue
        if date_to and day > date_to:
            continue
        matches.append(entry)

    if not matches:
        return "No matching entries."

    lines = []
    total = 0.0
    for entry in matches:
        hours = entry["hours"]
        if hours is not None:
            total += float(hours)
        shown = f"{hours} h" if hours is not None else "hours not given"
        lines.append(f"{entry['logged_at'][:10]} | {entry['project']} | {entry['description']} | {shown}")
    lines.append(f"{len(matches)} entries. Total hours where given: {total:g}")
    return "\n".join(lines)


def update_last_entry(project=None, description=None, hours=None):
    """Change the most recent entry. Only the fields given are changed."""
    entries = read_entries()
    if not entries:
        return "Error: the log is empty, so there is nothing to update."
    last = entries[-1]
    if project is not None:
        last["project"] = project
    if description is not None:
        last["description"] = description
    if hours is not None:
        last["hours"] = hours
    write_entries(entries)
    return f"Updated the last entry: {last['project']} | {last['description']} | {last['hours']} h"


def show_log():
    """Print the demo log, one line per entry."""
    for e in read_entries():
        print(f"  {e['logged_at'][:16]} | {e['project']} | {e['description']} | {e['hours']}")


reset_log()
print(add_entry("Herd Planner", "Testing by hand", 1))
print(update_last_entry(hours=2))
print()
show_log()
reset_log()

Saved entry for Herd Planner.
Updated the last entry: Herd Planner | Testing by hand | 2 h

  2026-11-15T09:00 | Cornerpost Watch | Tuned the cloud mask | 1
  2026-11-15T14:00 | Agentic Engineering Worklog | Ran lesson 3 | None
  2026-11-16T08:30 | Cornerpost Watch | Fixed the alert email bug | 2
  2026-11-16T10:00 | Herd Planner | Added vaccine records | 1.5
  2026-11-16T22:07 | Herd Planner | Testing by hand | 2


### The code, line by line

- `read_entries()` reads the whole log, as in lesson 3.
- `write_entries(entries)` is new: it **replaces** the whole file with a list of entries. Changing one entry means rewriting the file, because a JSON Lines file can only be added to at the end.
- `add_entry` and `search_entries` are lesson 3's.
- `update_last_entry(project=None, description=None, hours=None)`:
    - reads all entries. If there are none, it returns an error message.
    - `last = entries[-1]` takes the most recent one (`[-1]` means the last item).
    - it changes **only the parts it was given**. `if hours is not None:` means "only if hours were actually passed".
    - it saves the whole list back, and returns a description of the updated entry.
- `show_log()` prints the log neatly, one entry per line.
- The test at the bottom adds an entry with 1 hour, corrects it to 2, prints the log, then resets it.

### How to read it

Your run printed:

```
Saved entry for Herd Planner.
Updated the last entry: Herd Planner | Testing by hand | 2 h
```

then the five entries, ending with `Herd Planner | Testing by hand | 2`. The entry was saved with 1 hour and corrected to **2**, and no second entry was added.

### What it means

Two design choices keep this tool safe:

- **It only changes the last entry.** The model can't accidentally edit something from last week. A correction almost always refers to the thing just said.
- **It only changes the parts it's given.** "Make that two hours" changes the hours and leaves the project and description alone.

A narrow tool like this limits how much damage a mistake can do. That's a good general rule for agents: give each tool the smallest power that does the job.

### What's next and why

Step 3 describes the new tool to the model, and adds a rule about corrections.

## Step 3: Describe the tools and write the instructions

### What we are looking for

A description that tells the model when a message is a correction, and one new rule in the system prompt.

In [3]:
ADD_ENTRY_TOOL = {
    "type": "function",
    "function": {
        "name": "add_entry",
        "description": "Save one new work log entry. Use this when the user describes work they did.",
        "parameters": {
            "type": "object",
            "properties": {
                "project": {"type": "string", "description": "Name of the project the work was for."},
                "description": {"type": "string", "description": "Short description of what was done."},
                "hours": {"type": "number", "description": "Hours spent. Leave this out if the user did not say."},
            },
            "required": ["project", "description"],
        },
    },
}

SEARCH_ENTRIES_TOOL = {
    "type": "function",
    "function": {
        "name": "search_entries",
        "description": (
            "Look up saved work log entries. Use this to answer questions about past work. "
            "Returns the matching entries and the total hours."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "project": {"type": "string", "description": "Only entries for this project. Leave out for all projects."},
                "date_from": {"type": "string", "description": "First day to include, as YYYY-MM-DD."},
                "date_to": {"type": "string", "description": "Last day to include, as YYYY-MM-DD."},
            },
            "required": [],
        },
    },
}

UPDATE_LAST_ENTRY_TOOL = {
    "type": "function",
    "function": {
        "name": "update_last_entry",
        "description": (
            "Correct the most recently saved entry. Use this when the user changes something they just logged, "
            "for example 'make that two hours' or 'that was for Herd Planner'. Only pass the fields that change."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "project": {"type": "string", "description": "The corrected project name."},
                "description": {"type": "string", "description": "The corrected description."},
                "hours": {"type": "number", "description": "The corrected number of hours."},
            },
            "required": [],
        },
    },
}

TOOLS = [ADD_ENTRY_TOOL, SEARCH_ENTRIES_TOOL, UPDATE_LAST_ENTRY_TOOL]
AVAILABLE_TOOLS = {
    "add_entry": add_entry,
    "search_entries": search_entries,
    "update_last_entry": update_last_entry,
}

KNOWN_PROJECTS = ["Cornerpost Watch", "Cornerpost Logic", "Herd Planner", "Agentic Engineering Worklog"]

SYSTEM_PROMPT = (
    "You are a work log assistant. "
    f"Today is {TODAY_S} ({TODAY.strftime('%A')}). "
    "When the user describes work they did, call add_entry. "
    "When the user asks about past work, call search_entries and answer from its result. "
    "If a request needs both, save first and search after. "
    "If the user corrects something they just logged, call update_last_entry instead of add_entry. "
    "Known projects: " + ", ".join(KNOWN_PROJECTS) + ". "
    "Use one of those names when the work matches one. "
    "Only include hours if the user stated them. Convert minutes to hours. "
    "If the message is not about work, answer normally without calling a tool."
)


def call_model(messages):
    """One call to the local model with all three tools on offer."""
    return ollama.chat(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        think=False,
        options={"temperature": 0},
    )


def run_tool(name, arguments):
    """Run one requested tool and return its text result, turning mistakes into error text."""
    if name not in AVAILABLE_TOOLS:
        return f"Error: there is no tool named {name}."
    try:
        return AVAILABLE_TOOLS[name](**arguments)
    except TypeError as error:
        return f"Error: {error}"


print("Tools on offer:", ", ".join(AVAILABLE_TOOLS))

Tools on offer: add_entry, search_entries, update_last_entry


### The code, line by line

- `ADD_ENTRY_TOOL` now says "Save one **new** work log entry", to separate it clearly from correcting.
- `UPDATE_LAST_ENTRY_TOOL`'s description includes two **examples**: "make that two hours" and "that was for Herd Planner". All its inputs are optional.
- `SYSTEM_PROMPT` adds: "If the user corrects something they just logged, call update_last_entry instead of add_entry."
- `call_model` offers all three tools.
- `run_tool(name, arguments)` is the error handling from lesson 3's loop, moved into its own function so both versions of the agent below can share it.

### How to read it

Your run printed `Tools on offer: add_entry, search_entries, update_last_entry`.

### What it means

Examples in a description are often the most effective part of it, because they show the model what kind of sentence should trigger the tool. "Make that two hours" doesn't mention correcting or updating, but the example teaches the model to recognize it.

### What's next and why

Step 4 tries a correction with lesson 3's agent, which has **no** conversation memory, to see what memory has to add.

## Step 4: A correction, with no conversation memory

### What we are looking for

What lesson 3's agent does with "Actually, make that two hours" when it can't see the first message. It was expected to add a second entry, or to ask what "that" means.

In [4]:
MAX_STEPS = 6


def run_agent(user_text, verbose=True):
    """Lesson 3's agent: a full loop, but no memory of earlier requests."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    for step in range(1, MAX_STEPS + 1):
        response = call_model(messages)
        tool_calls = response.message.tool_calls or []
        if not tool_calls:
            return response.message.content
        messages.append(response.message)
        for call in tool_calls:
            name, arguments = call.function.name, dict(call.function.arguments)
            result = run_tool(name, arguments)
            if verbose:
                print(f"  step {step}: {name}({arguments}) -> {result.splitlines()[0]}")
            messages.append({"role": "tool", "tool_name": name, "content": result})
    return f"Stopped after {MAX_STEPS} steps without a final answer."


reset_log()
for text in ["Log an hour on Herd Planner fixing the weight chart.", "Actually, make that two hours."]:
    print("You:  ", text)
    print("Model:", run_agent(text))
    print()

print("Demo log afterwards:")
show_log()

You:   Log an hour on Herd Planner fixing the weight chart.
  step 1: add_entry({'description': 'Fixing the weight chart', 'hours': 1, 'project': 'Herd Planner'}) -> Saved entry for Herd Planner.
Model: The entry for Herd Planner has been saved.

You:   Actually, make that two hours.
  step 1: update_last_entry({'hours': 2}) -> Updated the last entry: Herd Planner | Fixing the weight chart | 2 h
Model: The last entry has been updated to 2 hours for the Herd Planner project. Let me know if you need anything else!

Demo log afterwards:
  2026-11-15T09:00 | Cornerpost Watch | Tuned the cloud mask | 1
  2026-11-15T14:00 | Agentic Engineering Worklog | Ran lesson 3 | None
  2026-11-16T08:30 | Cornerpost Watch | Fixed the alert email bug | 2
  2026-11-16T10:00 | Herd Planner | Added vaccine records | 1.5
  2026-11-16T22:07 | Herd Planner | Fixing the weight chart | 2


### The code, line by line

- `run_agent` is lesson 3's agent, unchanged except that it uses `run_tool`. Each call builds a **brand-new** message list: the system prompt and one user message, nothing else.
- The loop at the bottom sends the two messages as two separate calls, then prints the demo log.

### How to read it

Your run printed:

```
You:   Log an hour on Herd Planner fixing the weight chart.
  step 1: add_entry({'description': 'Fixing the weight chart', 'hours': 1, 'project': 'Herd Planner'}) -> Saved entry for Herd Planner.

You:   Actually, make that two hours.
  step 1: update_last_entry({'hours': 2}) -> Updated the last entry: Herd Planner | Fixing the weight chart | 2 h
```

and the log ends with **one** Herd Planner entry of **2 hours**. It got it right, with no conversation memory.

### What it means

This wasn't the expected result, and the reason is worth understanding.

The second call only saw "Actually, make that two hours." It had no idea what "that" was. But it didn't need to know: the tool's description told it that "make that two hours" means "correct the last entry", so it called `update_last_entry` with `hours: 2`. The **tool** then found the last entry by reading the log file, which held the Herd Planner entry that had just been saved.

So the memory came from the **saved data**, not the conversation:

| Kind of memory | Where it lives | What it remembers |
|---|---|---|
| Conversation memory | The message list your code sends | What was *said* |
| Saved data | The log file | What was *done* |

For "fix the last thing I logged", the log file is enough, because "the last thing" is always its last line. This is a useful design lesson: **keeping important state in saved data, instead of relying on the model to remember it, makes an agent more reliable.** The file can't forget, misread, or run out of room.

Conversation memory is still needed for things that were said but never saved. Step 7 shows one.

### What's next and why

Step 5 builds conversation memory anyway, and Steps 6 to 9 test where it does and doesn't make a difference.

## Step 5: Conversation memory, with a limit

### What we are looking for

A `Conversation` that keeps recent turns and sends them with each new message, dropping the oldest when there are too many.

In [5]:
class Conversation:
    """An agent that remembers recent turns. A turn is one user message and everything that followed it."""

    def __init__(self, max_turns=5):
        self.max_turns = max_turns
        self.turns = []
        self.prompt_tokens = []  # how much text the model read on each turn's first call

    def recent_messages(self):
        """The messages from the last max_turns turns, flattened into one list."""
        kept = self.turns[-self.max_turns:] if self.max_turns > 0 else []
        return [message for turn in kept for message in turn]

    def send(self, user_text, verbose=True):
        """Handle one user message with memory of recent turns. Returns the reply and the tool calls made."""
        turn = [{"role": "user", "content": user_text}]
        calls_made = []
        reply = f"Stopped after {MAX_STEPS} steps without a final answer."

        for step in range(1, MAX_STEPS + 1):
            messages = [{"role": "system", "content": SYSTEM_PROMPT}] + self.recent_messages() + turn
            response = call_model(messages)
            if step == 1:
                self.prompt_tokens.append(response.prompt_eval_count)
            tool_calls = response.message.tool_calls or []

            if not tool_calls:
                reply = response.message.content
                break

            turn.append(response.message)
            for call in tool_calls:
                name, arguments = call.function.name, dict(call.function.arguments)
                result = run_tool(name, arguments)
                calls_made.append((name, arguments))
                if verbose:
                    print(f"  step {step}: {name}({arguments}) -> {result.splitlines()[0]}")
                turn.append({"role": "tool", "tool_name": name, "content": result})

        turn.append({"role": "assistant", "content": reply})
        self.turns.append(turn)
        return reply, calls_made


print("Conversation ready.")

Conversation ready.


### The code, line by line

This is a **class**: a way of bundling data with the functions that use it. A `Conversation` keeps its own list of turns, and you can make as many separate conversations as you like.

- `__init__(self, max_turns=5)` runs when a conversation is created. It stores the limit and starts two empty lists:
    - `turns`: the conversation so far. A **turn** is one user message plus everything that followed it: tool requests, tool results and the reply.
    - `prompt_tokens`: how many tokens the model read on each turn, to show the cost of memory.
- `recent_messages()` returns the last `max_turns` turns as one flat list of messages. `self.turns[-self.max_turns:]` means "the last N items". With `max_turns=0` it returns nothing, meaning no memory.
- `send(user_text)` handles one message. It's lesson 3's loop with one change:

```python
messages = [{"role": "system", "content": SYSTEM_PROMPT}] + self.recent_messages() + turn
```

  The earlier turns go in front of the new one. That one line *is* the memory.
- After the loop, the finished turn, including the final reply, is added to `turns`.
- `response.prompt_eval_count` (from lesson 1) records how many tokens the model read.

### How to read it

Your run printed `Conversation ready.` This cell only defines the class.

### What it means

**Turns are kept whole.** A tool result only makes sense next to the request that asked for it. If the limit cut a turn in half, the model might see a result with no request, which confuses it. So the limit drops whole turns, oldest first.

### What's next and why

Step 6 repeats Step 4's correction with conversation memory switched on.

## Step 6: The same correction, with conversation memory

### What we are looking for

The same two messages, sent through one `Conversation` that remembers up to 5 turns.

In [6]:
reset_log()
chat = Conversation(max_turns=5)

for text in ["Log an hour on Herd Planner fixing the weight chart.", "Actually, make that two hours."]:
    print("You:  ", text)
    reply, calls = chat.send(text)
    print("Model:", reply)
    print()

print("Demo log afterwards:")
show_log()

You:   Log an hour on Herd Planner fixing the weight chart.
  step 1: add_entry({'description': 'Fixing the weight chart', 'hours': 1, 'project': 'Herd Planner'}) -> Saved entry for Herd Planner.
Model: The entry for Herd Planner has been saved.

You:   Actually, make that two hours.
  step 1: update_last_entry({'hours': 2}) -> Updated the last entry: Herd Planner | Fixing the weight chart | 2 h
Model: The last entry for Herd Planner has been updated to 2 hours for "Fixing the weight chart."

Demo log afterwards:
  2026-11-15T09:00 | Cornerpost Watch | Tuned the cloud mask | 1
  2026-11-15T14:00 | Agentic Engineering Worklog | Ran lesson 3 | None
  2026-11-16T08:30 | Cornerpost Watch | Fixed the alert email bug | 2
  2026-11-16T10:00 | Herd Planner | Added vaccine records | 1.5
  2026-11-16T22:08 | Herd Planner | Fixing the weight chart | 2


### The code, line by line

The log is reset, a new `Conversation` is created, and the two messages are sent through it, so the second one arrives with the first turn in front of it. Then the log is printed.

### How to read it

Your run printed the same tool calls as Step 4: `add_entry` with 1 hour, then `update_last_entry` with `hours: 2`. The log ends with one Herd Planner entry of 2 hours. This time the reply named the entry, "Fixing the weight chart", because the model could see the earlier turn.

### What it means

The result is the same as without memory, because Step 4 already showed this correction doesn't need it. That's an honest finding, not a failure. The memory version's reply is a little more informed, but the *log*, which is what matters, came out the same.

### What's next and why

Step 7 asks something that only conversation memory can answer, and measures what memory costs.

## Step 7: What memory adds, and what it costs

### What we are looking for

A five-message conversation, including one question that can only be answered from memory, and a record of how much text the model had to read each turn.

In [7]:
reset_log()
chat = Conversation(max_turns=5)

for text in [
    "Log an hour on Herd Planner fixing the weight chart.",
    "Actually, make that two hours.",
    "What project was that for?",
    "How many hours have I logged on Herd Planner today?",
    "Thanks. What's the capital of Colorado?",
]:
    reply, calls = chat.send(text, verbose=False)
    print(f"You:   {text}")
    print(f"Model: {reply.strip()[:200]}")
    print()

print("Text the model read on each turn (tokens):", chat.prompt_tokens)

You:   Log an hour on Herd Planner fixing the weight chart.
Model: The entry for Herd Planner has been saved.

You:   Actually, make that two hours.
Model: The last entry for Herd Planner has been updated to 2 hours for "Fixing the weight chart."

You:   What project was that for?
Model: That was for the Herd Planner project.

You:   How many hours have I logged on Herd Planner today?
Model: You have logged 3.5 hours on the Herd Planner project today.

You:   Thanks. What's the capital of Colorado?
Model: The capital of Colorado is Denver.

Text the model read on each turn (tokens): [573, 652, 741, 772, 931]


### The code, line by line

One `Conversation` handles five messages in a row, printing each message and the start of each reply. The last line prints `chat.prompt_tokens`: the tokens read on each turn.

### How to read it

Your run printed:

| You said | Model replied | Needed |
|---|---|---|
| Log an hour on Herd Planner fixing the weight chart. | Saved. | a tool |
| Actually, make that two hours. | Updated to 2 hours. | a tool (no memory needed, as Step 4 showed) |
| What project was that for? | That was for the Herd Planner project. | **conversation memory** |
| How many hours have I logged on Herd Planner today? | 3.5 hours. | a search (1.5 + 2) |
| Thanks. What's the capital of Colorado? | Denver. | nothing |

```
Text the model read on each turn (tokens): [573, 652, 741, 772, 931]
```

### What it means

**"What project was that for?" is the one that needed conversation memory.** No tool can answer it, because it's a question about the conversation itself, not about saved data. The model answered correctly because the earlier turns were in its message list.

**The token numbers show the cost:**

- **573 on the first turn**, before there was any history. That's the system prompt plus the descriptions of all three tools. Tools aren't free: their descriptions are re-read on every call.
- **Each turn adds roughly 30 to 160 tokens**, as the conversation grows: 573, 652, 741, 772, 931.
- By turn 5, the model was reading about **60% more** than on turn 1, for the same kind of short question.

At 20 tokens per second for writing, reading is much faster, so this cost is small here. But it grows with every turn, and a full day of logging would keep growing. That's why `max_turns` exists.

### What's next and why

Step 8 tests what happens when the limit is set to remember nothing.

## Step 8: A limit that's too small

### What we are looking for

Whether "my main project" still means Cornerpost Watch when the agent remembers nothing (`max_turns=0`), compared with remembering 5 turns.

In [8]:
MESSAGES = [
    "My main project this week is Cornerpost Watch.",
    "How many days are in a leap year?",
    "Log 3 hours on my main project fixing the map.",
]

for limit in [0, 5]:
    reset_log()
    chat = Conversation(max_turns=limit)
    print(f"max_turns = {limit}")
    for text in MESSAGES:
        reply, calls = chat.send(text, verbose=False)
        for name, arguments in calls:
            print(f"  [tool] {name}({arguments})")
    print("  Last entry in the log:", read_entries()[-1]["project"], "|", read_entries()[-1]["hours"], "h")
    print()

max_turns = 0
  [tool] add_entry({'description': 'Fixing the map', 'hours': 3, 'project': 'Cornerpost Watch'})
  Last entry in the log: Cornerpost Watch | 3 h

max_turns = 5
  [tool] add_entry({'description': 'Fixing the map', 'hours': 3, 'project': 'Cornerpost Watch'})
  Last entry in the log: Cornerpost Watch | 3 h



### The code, line by line

Three messages: one stating your main project, one unrelated question, and one using the phrase "my main project". They're sent twice, with `max_turns=0` and then `max_turns=5`, and the last entry in the log is printed each time.

### How to read it

Your run printed:

```
max_turns = 0
  [tool] add_entry({'description': 'Fixing the map', 'hours': 3, 'project': 'Cornerpost Watch'})
  Last entry in the log: Cornerpost Watch | 3 h

max_turns = 5
  [tool] add_entry({'description': 'Fixing the map', 'hours': 3, 'project': 'Cornerpost Watch'})
  Last entry in the log: Cornerpost Watch | 3 h
```

**Both** got Cornerpost Watch, including the version that remembered nothing.

### What it means

With `max_turns = 0`, the model never saw "My main project this week is Cornerpost Watch." So it **guessed**, and the guess happened to be right. There are two likely reasons, and this test can't tell them apart:

- **Cornerpost Watch is the first name in the known-projects list**, so it's the obvious default pick.
- **"Fixing the map" sounds like Cornerpost Watch,** which is a satellite and mapping project.

Either way, this test **didn't test memory**, because it could be passed without any. It's a good example of a common mistake in evaluation: a test that passes for the wrong reason. The fix is to choose a main project the model wouldn't guess, such as Herd Planner, with a task that doesn't hint at it. Exercise 2 does this.

### What's next and why

Step 9 runs the full test set. Given Steps 4 and 8, its results need a careful reading.

## Step 9: Measure it

### What we are looking for

Scores on six short conversations, each starting from the same four entries:

- **Log:** at the end, are the four starting entries unchanged, and are the new entries exactly right?
- **Answer:** where a final answer is expected, does it contain the right fact?

In [9]:
TEST_CONVERSATIONS = [
    {"messages": ["Log an hour on Herd Planner fixing the weight chart.", "Actually, make that two hours."],
     "new": [("Herd Planner", 2)], "answer": []},
    {"messages": ["Spent 45 minutes on Cornerpost Watch.", "Sorry, that was Cornerpost Logic, not Watch."],
     "new": [("Cornerpost Logic", 0.75)], "answer": []},
    {"messages": ["Log 2 hours on Cornerpost Watch writing alert tests.", "Also log an hour on Herd Planner."],
     "new": [("Cornerpost Watch", 2), ("Herd Planner", 1)], "answer": []},
    {"messages": ["My main project this week is Cornerpost Watch.", "Log 3 hours on my main project fixing the map."],
     "new": [("Cornerpost Watch", 3)], "answer": []},
    {"messages": ["Log an hour on Herd Planner.", "How many hours is that on Herd Planner today?"],
     "new": [("Herd Planner", 1)], "answer": [r"\b2\.5\b"]},
    {"messages": ["What is the capital of Colorado?", "And what about Wyoming?"],
     "new": [], "answer": [r"cheyenne"]},
]


def same_hours(got, expected):
    try:
        return float(got) == float(expected)
    except (TypeError, ValueError):
        return False


scores = {"log": 0, "answer": 0}
for case in TEST_CONVERSATIONS:
    reset_log()
    chat = Conversation(max_turns=5)
    tools_used = []
    for text in case["messages"]:
        reply, calls = chat.send(text, verbose=False)
        tools_used += [name for name, _ in calls]

    entries = read_entries()
    start_ok = entries[:len(STARTING_ENTRIES)] == STARTING_ENTRIES
    new = [(e["project"], e["hours"]) for e in entries[len(STARTING_ENTRIES):]]
    new_ok = len(new) == len(case["new"]) and all(
        g[0] == w[0] and same_hours(g[1], w[1]) for g, w in zip(new, case["new"])
    )
    log_ok = start_ok and new_ok
    answer_ok = all(re.search(p, reply or "", re.IGNORECASE) for p in case["answer"])

    scores["log"] += log_ok
    scores["answer"] += answer_ok

    print(" / ".join(case["messages"]))
    print(f"   log: {'right' if log_ok else 'WRONG'} {new}   answer: {'right' if answer_ok else 'WRONG'}   tools: {tools_used}")
    print(f"   last reply: {(reply or '').strip()[:150]}")
    print()

reset_log()
total = len(TEST_CONVERSATIONS)
print(f"Log right:    {scores['log']} of {total}")
print(f"Answer right: {scores['answer']} of {total}")

Log an hour on Herd Planner fixing the weight chart. / Actually, make that two hours.
   log: right [('Herd Planner', 2)]   answer: right   tools: ['add_entry', 'update_last_entry']
   last reply: The last entry for Herd Planner has been updated to 2 hours for "Fixing the weight chart."

Spent 45 minutes on Cornerpost Watch. / Sorry, that was Cornerpost Logic, not Watch.
   log: right [('Cornerpost Logic', 0.75)]   answer: right   tools: ['add_entry', 'update_last_entry']
   last reply: The last entry has been updated to: **Cornerpost Logic | Spent 45 minutes on Cornerpost Logic | 0.75 h**.

Log 2 hours on Cornerpost Watch writing alert tests. / Also log an hour on Herd Planner.
   log: right [('Cornerpost Watch', 2), ('Herd Planner', 1)]   answer: right   tools: ['add_entry', 'add_entry']
   last reply: The entry for Herd Planner has been saved.

My main project this week is Cornerpost Watch. / Log 3 hours on my main project fixing the map.
   log: right [('Cornerpost Watch', 3)]   an

### The code, line by line

- `TEST_CONVERSATIONS` lists six conversations. Each has its messages, the entries that should be added (`new`), and patterns the last reply must contain (`answer`).
- For each one, the log is reset, a fresh `Conversation` (5 turns) handles the messages, and the results are checked:
    - `start_ok`: the four starting entries are exactly unchanged. A correction that edited the wrong entry fails here.
    - `new_ok`: the right number of new entries, each with the right project and hours.
    - `answer_ok`: every pattern is found in the last reply.
- The log is reset once more at the end.

### How to read it

Your run printed:

```
Log right:    6 of 6
Answer right: 6 of 6
```

| Conversation | Result | Did it need conversation memory? |
|---|---|---|
| Log an hour on Herd Planner / Actually, make that two hours | right: one entry, 2 h | **No.** Step 4 passed this without memory. |
| Spent 45 minutes on Cornerpost Watch / Sorry, that was Cornerpost Logic | right: Cornerpost Logic, 0.75 h | **Probably not.** The second message names the new project, and the tool finds the last entry. |
| Log 2 hours on Cornerpost Watch / Also log an hour on Herd Planner | right: two entries | **No.** Each message stands on its own. This one checks that "also" adds rather than corrects. |
| My main project is Cornerpost Watch / Log 3 hours on my main project | right: Cornerpost Watch, 3 h | **Unclear.** Step 8 passed this without memory. |
| Log an hour on Herd Planner / How many hours is that today? | right: 2.5 | **No.** A search answers it from the log. |
| What is the capital of Colorado? / And what about Wyoming? | right: Cheyenne | **Yes.** "What about Wyoming?" means nothing without the first question. |

### What it means

The agent passed every test, and the log was right every time, including the strict check that the starting entries weren't touched. That's a real result: **the agent handles corrections and follow-ups correctly.**

But the table shows that **only one test clearly needs conversation memory**, and two more are uncertain. Most of these conversations would pass without it, because the tools and the log file carry what's needed. So 6 of 6 shows the agent works; it doesn't prove that *memory* is what makes it work.

The way to find out is a **control**: run the same tests with memory switched off (`max_turns=0`) and compare. Any test that still passes isn't testing memory. This is the same idea as a control group in data science: to know what an ingredient does, compare against a version without it. Exercise 3 does this.

## Exercises

1. **Correct something older.** In one conversation, log two entries, then say "change the first one to 3 hours". `update_last_entry` can only change the last one. What does the agent do: refuse, explain, or change the wrong entry?
2. **A main project the model wouldn't guess.** In Step 8, change the first message to "My main project this week is Herd Planner." and the third to "Log 3 hours on my main project." Run it again. Does `max_turns = 0` still get it right?
3. **Run a control.** Copy the Step 9 cell, change `Conversation(max_turns=5)` to `Conversation(max_turns=0)`, and run it. Which conversations still pass without memory? Those are the ones that aren't testing memory.
4. **Watch the cost.** In Step 7, add five more messages, and look at how `prompt_tokens` grows.

## What it all means

- **A model has no memory of its own.** Conversation memory is your code keeping the messages and sending them again.
- **There are two kinds of memory:** the conversation (what was *said*) and saved data (what was *done*). Many follow-ups only need saved data, and saved data is more reliable.
- **Memory has a cost.** Every remembered turn, and every tool description, is re-read on every call. Your run went from 573 to 931 tokens in five turns.
- **Trim whole turns,** so a tool result is never sent without the request that asked for it.
- **A narrow tool** (only the *last* entry) makes mistakes smaller.
- **A passing test isn't proof.** Check that each test can only pass for the reason you think. A control, run with the feature switched off, shows which tests really measure it.

## Words used in this lesson

| Word | Meaning |
|---|---|
| Conversation memory | Earlier messages, kept by your code and sent again with each new one. |
| Saved data / state | Information stored outside the conversation, such as the log file. |
| Turn | One user message plus everything that followed: tool calls, results and the reply. |
| `max_turns` | How many earlier turns the conversation keeps. |
| Class | A way of bundling data with the functions that use it, such as `Conversation`. |
| Control | The same test with one thing switched off, to see what that thing actually does. |
| Demo log | A separate log file with known entries, reset before each demonstration. |

## What's next and why

**Lesson 5: A real job.** The agent works inside a notebook. Lesson 5 turns it into a small program you can run every day from the terminal, logging to your real work log, with everything from lessons 1 to 4 in one place. Its tests will include a control. That program is the start of the productivity desktop app.